In [1]:
#@title Import Modules
!pip install "httpx[http2,brotli]" parsel
!pip install loguru

# import modules
import re
import httpx
import json
import random
import string
import difflib
import math
import numpy as np
import requests
import pandas as pd
from google.colab import files
from loguru import logger as log

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.0 MB/s eta 0:00:00


In [2]:
# To avoid being instantly blocked we'll be using request headers that
# mimic Chrome browser on Windows
BASE_HEADERS = {
    "authority": "www.tripadvisor.com",
    "accept": "application/json", # Explicitly setting application/json for GraphQL
    "accept-encoding": "gzip, deflate, br, zstd",
    "accept-language": "en-US,en;q=0.9,id;q=0.8,fa;q=0.7,ar;q=0.6,ms;q=0.5,ja;q=0.4,es;q=0.3",
    "content-type": "application/json",
    "cookie": "TAUnique=%1%enc%3ALezWdGmjvm7%2Fyz5vAHmDMnQWX1FENGAF%2BjFAX07m8QI0Z3D5GE4M5PQd%2FfSJ%2BBSxNox8JbUSTxk%3D; TADCID=9HhQM_Lz5aOaFrQeABQChrLCR-aOT_K5vlfY_RXuc43Jel6WpagCa6ZcTVHL6itL8Uo-GkGmKvlBDelXi9VxVNLR3dR3fEh2S0g; ServerPool=B; PMC=V2*MS.67*MD.20260523*LD.20260523; TART=%1%enc%3AZTG1BOzIKHKsNnZOaTLI5qCWUs8vRnzmj%2BsSsAzXYlwdcyLxjkVWg3SC3DIF9HIcSyMvPHMDuZw%3D; TATrkConsent=eyJvdXQiOiIiLCJpbiI6IkFMTCJ9; TAReturnTo=%1%%2Fbusiness; __hs_cookie_cat_pref=1:true_2:true_3:true; TASID=131251635312FCEF8931F02A0C5982D2; TAUD=; datadome=8mBQCQT2w~5vmJBPXBM1YcKnDZ1O9K3OdmTxb~tIU8_5Dfhl3hbS0ZAGv3tfwhc4eMXZ8UB4XlU_UGBcOdzbMn0tuX5BovMtQQlfr4owitx69nOVilUoi7LiLUPg5BUr; OptanonConsent=isGpcEnabled=0&datestamp=Sat+May+23+2026+22%3A41%3A01+GMT%2B0800+(China+Standard+Time)&version=202601.2.0&browserGpcFlag=0&isIABGlobal=false&hosts=&consentId=790b6407-be14-417a-9c04-7230366271de&interactionCount=0&isAnonUser=1&prevHadToken=0&landingPath=NotLandingPage&groups=C0001%3A1%2CC0002%3A1%2CC0003%3A1%2CC0004%3A1&AwaitingReconsent=false; __vt=-8Erw-M4uAUEqmbNABQCT24E-H_BQo6gx1APGQJPtz2nYVlfIENULkfBYldlKYB9LtNbewyrK3DQXr7iOqiZDTe0-3UKQetkbv0VfildBeRCl5Thj-FohLi_1nyUcSnVdUbEB91fIfrP2w_nKd_WdZ93Ng",
    "origin": "https://www.tripadvisor.com",
    "priority": "u=1, i",
    "referer": "https://www.tripadvisor.com/Search?q=spa+nashville&geo=55229&ssrc=a&searchNearby=false&searchSessionId=0001beae495bf695.ssid&offset=0",
    "sec-ch-device-memory": "4",
    "sec-ch-ua": "\"Chromium\";v=\"148\", \"Microsoft Edge\";v=\"148\", \"Not/A)Brand\";v=\"99\"",
    "sec-ch-ua-arch": "\"x86\"",
    "sec-ch-ua-full-version-list": "\"Chromium\";v=\"148.0.7778.168\", \"Microsoft Edge\";v=\"148.0.3967.70\", \"Not/A)Brand\";v=\"99.0.0.0\"",
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-model": "\"\"",
    "sec-ch-ua-platform": "\"Windows\"",
    "sec-fetch-dest": "empty",
    "sec-fetch-mode": "same-origin",
    "sec-fetch-site": "same-origin",
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36 Edg/148.0.0.0"
}
# start HTTP session client with our headers and HTTP2
client = httpx.AsyncClient(
    http2=True,  # http2 connections are significantly less likely to get blocked
    headers=BASE_HEADERS,
    timeout=httpx.Timeout(15.0),
    limits=httpx.Limits(max_connections=5),
)

In [ ]:
# payload = [
#     {
#         "variables":{
#             "request":{
#                 "query":"banjarmasin",
#                 "limit":10,
#                 "scope":"WORLDWIDE",
#                 "locale":"en-US",
#                 "scopeGeoId":1,
#                 "searchCenter":None,
#                 "types":["LOCATION","RESCUE_RESULT"],
#                 "locationTypes":["GEO","NEIGHBORHOOD","ACCOMMODATION"],
#                 "userId":None,
#                 "context":{
#                     "searchSessionId":"0002d1b71c31f8cc.ssid",
#                     "typeaheadId":"1752215195697",
#                     "uiOrigin":"homepage_faceted_search_HOTELS",
#                     "routeUid":"c50707db-8d80-4795-b53f-b37a5b036f26"
#                     },
#                 "enabledFeatures":[],
#                 "includeRecent":True,
#                 "includeNestedResults":True
#                 }
#             },
#         "extensions":{
#             "preRegisteredQueryId":"c2e5695e939386e4"
#             }
#         }
#     ]

In [5]:
# MediaAlbum
# payload = [
#     {
#       "variables": {
#         "albumId": -125,
#         "config": "apd",
#         "client": "apf",
#         "limit": 10,
#         "offset": 0,
#         "locationId": 33861026
#       },
#       "extensions": {
#         "preRegisteredQueryId": "9a7fd69b592bf69c"
#       }
#     }
# ]
# AttractionProductsFusion
# payload = [
#     {
#         "variables": {
#             "commerce": {
#                 "attractionCommerce": {
#                     "pax": [
#                         {
#                             "ageBand": "ADULT",
#                             "count": 2
#                         }
#                     ],
#                     "startDate": "2026-06-14",
#                     "endDate": "2026-06-14",
#                     "setByUser": True
#                 }
#             },
#             "currency": "USD",
#             "request": {
#                 "tracking": {
#                     "screenName": "AttractionProductsFusion",
#                     "pageviewUid": "0c04d870-a1b8-43d8-88a5-3efdc09a677d"
#                 },
#                 "routeParameters": {
#                     "geoId": 60763,
#                     "filters": [
#                         {
#                             "id": "detailId",
#                             "value": [
#                                 "105127"
#                             ]
#                         }
#                     ],
#                     "contentType": "attraction_product",
#                     "webVariant": "AttractionProductsFusion",
#                     "pagee": "28"
#                 },
#                 "updateToken": None
#             },
#             "sessionId": "6114A0B59416C7189867372AE1B017D8",
#             "tracking": {
#                 "screenName": "AttractionProductsFusion",
#                 "pageviewUid": "0c04d870-a1b8-43d8-88a5-3efdc09a677d"
#             },
#             "unitLength": "MILES"
#         },
#         "extensions": {
#             "preRegisteredQueryId": "44db135e467b5baa"
#         }
#     }
# ]

payload = [
    {
      "variables": {
        "request": {
          "tracking": {
            "screenName": "Attraction_Review",
            "pageviewUid": "33333384-dbf2-46a9-8075-d34a50a64d76"
          },
          "routeParameters": {
            "contentType": "attraction",
            "contentId": "105127"
          },
          "clientState": None,
          "updateToken": None
        },
        "commerce": {},
        "sessionId": "6114A0B59416C7189867372AE1B017D8",
        "tracking": {
          "screenName": "Attraction_Review",
          "pageviewUid": "33333384-dbf2-46a9-8075-d34a50a64d76"
        },
        "currency": "USD",
        "currentGeoPoint": None,
        "unitLength": "MILES"
      },
      "extensions": {
        "preRegisteredQueryId": "a1579600caae083f"
      }
    }
]

# we need to generate a random request ID for this request to succeed
random_request_id = "".join(
    random.choice(string.ascii_lowercase + string.digits) for i in range(180)
)

# set the headers
headers = {
    "X-Requested-By": random_request_id,
    "Referer": "https://www.tripadvisor.com/",
    "Origin": "https://www.tripadvisor.com",
}

# get the page with method POST and pass the payload and headers as the parameters
result = await client.post(
    url="https://www.tripadvisor.com/data/graphql/ids",
    json=payload,
    headers=headers,
)
data = json.loads(result.content)
data

[{'data': {'Result': [{'__typename': 'WebPresentation_QueryWebDetailResponse',
     'container': {'__typename': 'WebPresentation_WebDetailResponseContainer',
      'navTitle': 'Central Park',
      'pageTestVariants': None,
      'jsonLd': '{"@type":"LocalBusiness","name":"Central Park","url":"/Attraction_Review-g60763-d105127-Reviews-Central_Park-New_York_City_New_York.html","address":{"@type":"PostalAddress","streetAddress":"86th St Transverse Between Fifth Avenue and Central Park West","addressLocality":"New York City","addressRegion":"New York","addressCountry":"US","postalCode":"10019"},"aggregateRating":{"@type":"AggregateRating","ratingValue":"4.7","reviewCount":134447,"bestRating":5},"openingHours":"Mo-Su 06:00-01:00","image":"https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2f/c4/c8/4b/caption.jpg?w=1200&h=-1&s=1","telephone":"(212) 310-6600","@context":"https://schema.org","@id":"/Attraction_Review-g60763-d105127-Reviews-Central_Park-New_York_City_New_York.html","geo":

In [ ]:
# payload2 = [
#     {
#         "variables": {
#             "request": {
#                 "filters": {
#                     "dataTypes": [
#                         "LOCATION"
#                     ],
#                     "locationTypes": [
#                         "GEO",
#                         "ACCOMMODATION",
#                         "AIRLINE",
#                         "ATTRACTION",
#                         "ATTRACTION_PRODUCT",
#                         "EATERY",
#                         "NEIGHBORHOOD"
#                     ]
#                 },
#                 "locale": "en-US",
#                 "query": "gym",
#                 "offset": 0,
#                 "scope": {
#                     "locationId": 60763,
#                     "center": None
#                 },
#                 "locationIdsToExclude": [],
#                 "categoryFilterIds": [
#                     "ATTRACTIONS",
#                     "RESTAURANTS",
#                     "DESTINATIONS",
#                     "HOTELS"
#                 ],
#                 "additionalFields": [
#                     "SNIPPET",
#                     "MENTION_COUNT"
#                 ],
#                 "limit": 750
#             }
#         },
#         "extensions": {
#             "preRegisteredQueryId": "e04b9250204468c8"
#         }
#     }
# ]
payload2 = {
  "variables": {
    "request": {
      "tracking": {
        "screenName": "AttractionsFusion",
        "pageviewUid": "3333338c-1328-4219-8caa-8d6e0b658883"
      },
      "routeParameters": {
        "geoId": 60763,
        "filters": [
          {
            "id": "category",
            "value": [
              "61"
            ]
          }
        ],
        "contentType": "attraction",
        "webVariant": "AttractionsFusion",
        "pagee": "0"
      }
    },
    "sessionId": "ED40F9307BCC65AE606EE86016E8DEE6",
    "unitLength": "MILES",
    "currency": "USD",
    "currentGeoPoint": None,
    "mapSurface": False,
    "debug": False,
    "polling": False
  },
  "extensions": {
    "preRegisteredQueryId": "0a9ecd5c0d013d56"
  }
}

# we need to generate a random request ID for this request to succeed
random_request_id = "".join(
    random.choice(string.ascii_lowercase + string.digits) for i in range(180)
)

# set the headers
headers = {
    "X-Requested-By": random_request_id,
    "Referer": "https://www.tripadvisor.com/",
    "Origin": "https://www.tripadvisor.com",
}

# get the page with method POST and pass the payload and headers as the parameters
result = await client.post(
    url="https://www.tripadvisor.com/data/graphql/ids",
    json=payload2,
    headers=headers,
)
data = json.loads(result.content)
# data
# print(f"Total of result: {len(data[0]['data']['SERP_getSearchResultsList']['clusters'][0]['sections'][1]['results'])}")
# all_records = data[0]['data']['SERP_getSearchResultsList']['clusters'][0]['sections'][1]['results']
# filtered_records = [record for record in all_records if record.get('details') is not None]
# print(f"Total of filtered result: {len(filtered_records)}")
# filtered_records[0]

results = data['data']['Result'][0]['sections']
filtered_records = [record for record in results if record.get('__typename') == "WebPresentation_SingleFlexCardSection"]
print(f"Total of filtered result: {len(filtered_records)}")
filtered_records[0]

Total of filtered result: 30


{'__typename': 'WebPresentation_SingleFlexCardSection',
 'singleFlexCardContent': {'__typename': 'WebPresentation_FlexCard',
  'variant': 'STANDARD',
  'accessibilityString': {'__typename': 'WebPresentation_LocalizedString',
   'text': 'View details for NY Helicopter Tour: Manhattan Highlights'},
  'accommodationType': None,
  'amenityList': None,
  'poiAttachedProduct': None,
  'badge': None,
  'bubbleRating': {'__typename': 'WebPresentation_BubbleRating',
   'numberReviews': {'__typename': 'WebPresentation_LocalizedString',
    'text': '748'},
   'rating': 4.8,
   'reviewCount': 748},
  'cardLink': {'__typename': 'WebPresentation_InternalLink',
   'linkType': 'InternalLink',
   'trackingContext': 'server_card',
   'text': None,
   'webRoute': {'__typename': 'Routing_Route',
    'page': 'AttractionProductReview',
    'typedParams': {'__typename': 'Routing_AttractionProductReviewParameters'},
    'webLinkUrl': '/AttractionProductReview-g60763-d11462107-NY_Helicopter_Tour_Manhattan_High

In [ ]:
# data[0]['data']['SERP_getSearchResultsList']['clusters'][0]['sections'][1]['results']
# all_records = data[0]['data']['SERP_getSearchResultsList']['clusters'][0]['sections'][1]['results']
# filtered_records = [record for record in all_records if record.get('details') is not None]
# filtered_records[0]
data

[{'data': {'SERP_getSearchResultsList': {'trackingInfo': None,
    'clusters': [{'sections': [{'__typename': 'SERP_CategoryFiltersSection'},
       {'__typename': 'SERP_PagedSearchResultsSection',
        'count': 1712,
        'results': [{'__typename': 'SERP_LocationResult',
          'locationType': 'ATTRACTION',
          'locationId': 27049266,
          'snippet': None,
          'mentionCount': None,
          'details': {'defaultUrl': '/Attraction_Review-g32411-d27049266-Reviews-Fremont_Day_Spa-Fremont_California.html',
           'locationId': 27049266,
           'localizedName': 'Fremont Day Spa',
           'locationDescription': None,
           'locationV2': {'names': {'longOnlyHierarchyTypeaheadV2': 'Fremont, California'}},
           'thumbnail': None,
           'reviewSummary': {'rating': 0, 'count': 0},
           'detail': {'__typename': 'Attraction'}}},
         {'__typename': 'SERP_LocationResult',
          'locationType': 'ATTRACTION',
          'locationId': 27

In [ ]:
all_records = data[0]['data']['SERP_getSearchResultsList']['clusters'][0]['sections'][1]['results']
filtered_records = [record for record in all_records if record.get('details') is not None]

display(filtered_records)

In [ ]:
cities = [
  "New York, New York", "Los Angeles, California", "Chicago, Illinois", "Houston, Texas", "Phoenix, Arizona", "Philadelphia, Pennsylvania", "San Antonio, Texas", "San Diego, California", "Dallas, Texas", "Jacksonville, Florida", "Fort Worth, Texas", "San Jose, California", "Austin, Texas", "Charlotte, North Carolina", "Columbus, Ohio", "Indianapolis, Indiana", "San Francisco, California", "Seattle, Washington", "Denver, Colorado", "Oklahoma City, Oklahoma", "Nashville, Tennessee", "Washington, District of Columbia", "El Paso, Texas", "Las Vegas, Nevada", "Boston, Massachusetts", "Portland, Oregon", "Detroit, Michigan", "Louisville, Kentucky", "Memphis, Tennessee", "Baltimore, Maryland", "Milwaukee, Wisconsin", "Albuquerque, New Mexico", "Fresno, California", "Tucson, Arizona", "Sacramento, California", "Mesa, Arizona", "Kansas City, Missouri", "Atlanta, Georgia", "Colorado Springs, Colorado", "Omaha, Nebraska", "Raleigh, North Carolina", "Virginia Beach, Virginia", "Long Beach, California", "Miami, Florida", "Oakland, California", "Minneapolis, Minnesota", "Tulsa, Oklahoma", "Bakersfield, California", "Wichita, Kansas", "Arlington, Texas", "Aurora, Colorado", "Tampa, Florida", "New Orleans, Louisiana", "Cleveland, Ohio", "Anaheim, California", "Honolulu, Hawaii", "Henderson, Nevada",
  "Stockton, California", "Riverside, California", "Lexington, Kentucky", "Corpus Christi, Texas", "Orlando, Florida", "Santa Ana, California", "Cincinnati, Ohio", "Irvine, California", "St. Paul, Minnesota", "Pittsburgh, Pennsylvania", "St. Louis, Missouri", "Greensboro, North Carolina", "Jersey City, New Jersey", "Anchorage, Alaska", "Lincoln, Nebraska", "Plano, Texas", "Durham, North Carolina", "Buffalo, New York", "Chandler, Arizona", "Chula Vista, California", "Toledo, Ohio", "Madison, Wisconsin", "Gilbert, Arizona", "Reno, Nevada", "Fort Wayne, Indiana", "North Las Vegas, Nevada", "St. Petersburg, Florida", "Lubbock, Texas", "Irving, Texas", "Laredo, Texas", "Winston-Salem, North Carolina", "Chesapeake, Virginia", "Glendale, Arizona", "Garland, Texas", "Scottsdale, Arizona", "Norfolk, Virginia", "Boise, Idaho", "Fremont, California", "Spokane, Washington", "Santa Clarita, California", "Baton Rouge, Louisiana", "Richmond, Virginia", "Hialeah, Florida", "Newark, New Jersey", "McKinney, Texas", "Glendale, California", "San Bernardino, California", "Tacoma, Washington", "Salt Lake City, Utah", "Huntington Beach, California", "Des Moines, Iowa", "Frisco, Texas", "Grand Rapids, Michigan", "Moreno Valley, California", "Modesto, California", "Huntsville, Alabama", "Worcester, Massachusetts",
  "Rochester, New York", "Yonkers, New York", "Montgomery, Alabama", "Fayetteville, North Carolina", "Augusta, Georgia", "Amarillo, Texas", "Mobile, Alabama", "Little Rock, Arkansas", "Columbus, Georgia", "Oxnard, California", "Fontana, California", "Knoxville, Tennessee", "Grand Prairie, Texas", "Port St. Lucie, Florida", "Tallahassee, Florida", "Tempe, Arizona", "Cape Coral, Florida", "Overland Park, Kansas", "Brownsville, Texas", "Shreveport, Louisiana", "Birmingham, Alabama", "Newport News, Virginia", "Providence, Rhode Island", "Rancho Cucamonga, California", "Chattanooga, Tennessee", "Santa Rosa, California", "Oceanside, California", "Sioux Falls, South Dakota", "Vancouver, Washington", "Ontario, California", "Pembroke Pines, Florida"
]

In [ ]:
#@title Location ID Finder 2

import random
import string
import json

all_results = {}

for city in cities:
    print(f"Processing: {city}")

    new_payload = [{
        "variables": {
            "request": {
                "query": city,
                "limit": 10,
                "scope": "WORLDWIDE",
                "locale": "en-US",
                "scopeGeoId": 1,
                "searchCenter": None,
                "types": ["LOCATION", "QUERY_SUGGESTION", "RESCUE_RESULT"],
                "locationTypes": [
                    "GEO", "AIRPORT", "ACCOMMODATION", "ATTRACTION", "ATTRACTION_PRODUCT",
                    "EATERY", "NEIGHBORHOOD", "AIRLINE", "SHOPPING", "UNIVERSITY", "GENERAL_HOSPITAL",
                    "PORT", "FERRY", "CORPORATION", "SHIP", "CRUISE_LINE", "CAR_RENTAL_OFFICE"
                ],
                "userId": None,
                "context": {
                    "searchSessionId": "0015d9fc8600f84a.ssid",
                    "typeaheadId": "1781055485709",
                    "uiOrigin": "homepage_faceted_search_all",
                    "routeUid": "dd152eca-1611-4590-aad7-d1a20ad8d017"
                },
                "enabledFeatures": ["articles"],
                "includeRecent": True,
                "includeNestedResults": True
            },
            "locale": "en-US",
            "linkType": "",
            "includeGeoRoute": False,
            "includeLegacyParentHierarchy": False,
            "includeNaturalParentHierarchy": False
        },
        "extensions": {
            "preRegisteredQueryId": "c34966eb87c4a1cb"
        }
    }]

    random_request_id = "".join(
        random.choice(string.ascii_lowercase + string.digits) for i in range(180)
    )

    headers = {
        "X-Requested-By": random_request_id,
        "Referer": "https://www.tripadvisor.com/",
        "Origin": "https://www.tripadvisor.com",
    }

    try:
        result = await client.post(
            url="https://www.tripadvisor.com/data/graphql/ids",
            json=new_payload,
            headers=headers,
        )
        new_data = json.loads(result.content)

        locations = new_data[0]['data']['Typeahead_autocomplete']['results']
        print(f"  Total results: {len(locations)}")

        first_result = locations[0]

        city_output = {
            "location_id": first_result.get("locationId"),
            "localizedName": first_result.get("contentLocationNode", {}).get("detail", {}).get("info", {}).get("localizedName"),
            "contextualConciseParentHierarchy": first_result.get("contentLocationNode", {}).get("detail", {}).get("info", {}).get("contextualConciseParentHierarchy"),
            "latitude": first_result.get("contentLocationNode", {}).get("detail", {}).get("info", {}).get("latitude"),
            "longitude": first_result.get("contentLocationNode", {}).get("detail", {}).get("info", {}).get("longitude"),
            "primaryRoute": first_result.get("contentLocationNode", {}).get("detail", {}).get("info", {}).get("primaryRoute", {}).get("webLinkUrl"),
            "nested_results": []  # fixed key name (was "nested_firs_results" / "nested_results" mismatch)
        }

        for nested_item in first_result.get("nestedResults", []):
            city_output["nested_results"].append({
                "title": nested_item.get("title"),
                "url": nested_item.get("route", {}).get("url"),
                "category": nested_item.get("buCategory")
            })

        all_results[city] = city_output

    except Exception as e:
        print(f"  ERROR for '{city}': {e}")
        all_results[city] = {"error": str(e)}

# Save to JSON file
output_path = "tripadvisor_results3.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

print(f"\nDone! Results saved to {output_path}")
print(json.dumps(all_results, indent=2, ensure_ascii=False))

Processing: New York, New York
  Total results: 8
Processing: Los Angeles, California
  Total results: 8
Processing: Chicago, Illinois
  Total results: 8
Processing: Houston, Texas
  Total results: 8
Processing: Phoenix, Arizona
  Total results: 8
Processing: Philadelphia, Pennsylvania
  Total results: 8
Processing: San Antonio, Texas
  Total results: 8
Processing: San Diego, California
  Total results: 8
Processing: Dallas, Texas
  Total results: 8
Processing: Jacksonville, Florida
  Total results: 8
Processing: Fort Worth, Texas
  Total results: 8
Processing: San Jose, California
  Total results: 8
Processing: Austin, Texas
  Total results: 8
Processing: Charlotte, North Carolina
  Total results: 8
Processing: Columbus, Ohio
  Total results: 8
Processing: Indianapolis, Indiana
  Total results: 8
Processing: San Francisco, California
  Total results: 8
Processing: Seattle, Washington
  Total results: 8
Processing: Denver, Colorado
  Total results: 8
Processing: Oklahoma City, Oklahoma

In [ ]:
#@title Search Business Entities

import random
import string
import json

geo_id = 60763  # change as needed
page_size = 30

# categories = {
    # 'Spas & Wellness': '40',
    # 'Museums': '49',
    # 'Nightlife': '20',
    # 'Sights & Landmarks': '47',
    # 'Fun & Games': '56',
    # 'Nature & Parks': '57',
    # 'Classes & Workshops': '41',
    # 'Boat Tours & Water Sports': '55',
    # 'Casinos & Gambling': '53',
    # 'Water & Amusement Parks': '52',
    # 'Zoos & Aquariums': '48'
    # 'Outdoor Actitivites': '61'
# }

categories = {
    'Attractions': 'true',
    # 'Tours': '42',
    # 'Day Trips': '63',
    # 'Outdoor Activities': '61',
    # 'Concerts & Shows': '58',
    # 'Food & Drink': '36',
    # 'Events': '62',
    # 'Classes & Workshops': '41',
    # 'Shopping': '26',
    # 'Transportation': '59',
    # 'Traveler Resources': '60'
}

for category_name, category_id in categories.items():
    print(f"\n{'='*50}")
    print(f"Category: {category_name} (id: {category_id})")
    print(f"{'='*50}")

    all_filtered_records = []
    page = 0

    while True:
        print(f"  Fetching page offset {page}...")

        payload = {
          "variables": {
            "request": {
              "tracking": {
                "screenName": "AttractionsFusion",
                "pageviewUid": "3333338c-1328-4219-8caa-8d6e0b658883"
              },
              "routeParameters": {
                "geoId": geo_id,
                "filters": [
                  {
                    "id": "category",
                    "value": [category_id]
                  }
                ],
                "contentType": "attraction",
                "webVariant": "AttractionsFusion",
                "pagee": str(page)
              },
            },
            "sessionId": "ED40F9307BCC65AE606EE86016E8DEE6",
            "unitLength": "MILES",
            "currency": "USD",
            "currentGeoPoint": None,
            "mapSurface": False,
            "debug": False,
            "polling": False
          },
          "extensions": {
            "preRegisteredQueryId": "0a9ecd5c0d013d56"
          }
        }

        random_request_id = "".join(
            random.choice(string.ascii_lowercase + string.digits) for i in range(180)
        )

        headers = {
            "X-Requested-By": random_request_id,
            "Referer": "https://www.tripadvisor.com/",
            "Origin": "https://www.tripadvisor.com",
        }

        try:
            result = await client.post(
                url="https://www.tripadvisor.com/data/graphql/ids",
                json=payload,
                headers=headers,
            )
            data = json.loads(result.content)
            results = data['data']['Result'][0]['sections']
            filtered_records = [
                record for record in results
                if record.get('__typename') == "WebPresentation_SingleFlexCardSection"
            ]

            print(f"    Found {len(filtered_records)} records on offset {page}")
            all_filtered_records.extend(filtered_records)

            # Stop if we got fewer than 30 results (last page)
            if len(filtered_records) < page_size:
                print(f"    Last page reached ({len(filtered_records)} < {page_size}). Stopping.")
                break

            page += page_size  # advance: 0 → 30 → 60 → ...

        except Exception as e:
            print(f"    ERROR on offset {page}: {e}")
            break

    # Save each category to its own JSON file
    safe_category_name = category_name.replace(' ', '_').replace('&', 'and').replace('/', '-')
    output_path = f"tripadvisor_{safe_category_name}_{geo_id}.json"
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(all_filtered_records, f, indent=2, ensure_ascii=False)

    print(f"  Saved {len(all_filtered_records)} total records → {output_path}")

print(f"\n✅ All categories done!")

In [ ]:
import random
import string
import json

page_size = 30

categories = {
    'Attraction': 'true',  # special case → empty filters
    'Spas & Wellness': '40',
    'Museums': '49',
    'Nightlife': '20',
    'Sights & Landmarks': '47',
    'Fun & Games': '56',
    'Nature & Parks': '57',
    'Classes & Workshops': '41',
    'Boat Tours & Water Sports': '55',
    'Casinos & Gambling': '53',
    'Water & Amusement Parks': '52',
    'Zoos & Aquariums': '48'
}

# Load cities from file
with open("/content/150-cities-locationId.json", "r", encoding="utf-8") as f:
    cities_data = json.load(f)

for city_name, city_info in cities_data.items():
    geo_id = city_info["location_id"]
    localized_name = city_info.get("localizedName", city_name)

    print(f"\n{'#'*60}")
    print(f"City: {localized_name} (geo_id: {geo_id})")
    print(f"{'#'*60}")

    for category_name, category_value in categories.items():
        print(f"\n  {'='*50}")
        print(f"  Category: {category_name}")
        print(f"  {'='*50}")

        all_filtered_records = []
        page = 0

        # Requirement 1: if category value is 'true', use empty filters
        if category_value == 'true':
            filters = []
        else:
            filters = [{"id": "category", "value": [category_value]}]

        while True:
            print(f"    Fetching page offset {page}...")

            payload2 = {
                "variables": {
                    "request": {
                        "tracking": {
                            "screenName": "AttractionsFusion",
                            "pageviewUid": "333333ba-0c8e-4165-80f0-246118a6e5ef"
                        },
                        "routeParameters": {
                            "geoId": geo_id,
                            "filters": filters,       # empty [] or category filter
                            "contentType": "attraction",
                            "webVariant": "AttractionsFusion",
                            "pagee": str(page)
                        }
                    },
                    "sessionId": "239BCDD736C812FFBA1007D05B2D7008",
                    "unitLength": "MILES",
                    "currency": "USD",
                    "mapSurface": False,
                    "debug": False,
                    "polling": False
                },
                "extensions": {
                    "preRegisteredQueryId": "0a9ecd5c0d013d56"
                }
            }

            random_request_id = "".join(
                random.choice(string.ascii_lowercase + string.digits) for i in range(180)
            )

            headers = {
                "X-Requested-By": random_request_id,
                "Referer": "https://www.tripadvisor.com/",
                "Origin": "https://www.tripadvisor.com",
            }

            try:
                result = await client.post(
                    url="https://www.tripadvisor.com/data/graphql/ids",
                    json=payload2,
                    headers=headers,
                )
                data = json.loads(result.content)
                results = data['data']['Result'][0]['sections']
                filtered_records = [
                    record for record in results
                    if record.get('__typename') == "WebPresentation_SingleFlexCardSection"
                ]

                print(f"      Found {len(filtered_records)} records on offset {page}")
                all_filtered_records.extend(filtered_records)

                # Stop if fewer than 30 results — last page
                if len(filtered_records) < page_size:
                    print(f"      Last page reached ({len(filtered_records)} < {page_size}). Stopping.")
                    break

                page += page_size  # advance: 0 → 30 → 60 → ...

            except Exception as e:
                print(f"      ERROR on offset {page}: {e}")
                break

        # Requirement 3: save as "Los Angeles_Attraction.json"
        safe_category_name = category_name.replace(' ', '_').replace('&', 'and').replace('/', '-')
        output_path = f"{localized_name}_{safe_category_name}.json"
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(all_filtered_records, f, indent=2, ensure_ascii=False)

        print(f"  Saved {len(all_filtered_records)} total records → {output_path}")

print(f"\n✅ All cities and categories done!")


############################################################
City: New York City (geo_id: 60763)
############################################################

  Category: Attraction
    Fetching page offset 0...
      Found 30 records on offset 0
    Fetching page offset 30...
      Found 30 records on offset 30
    Fetching page offset 60...
      Found 30 records on offset 60
    Fetching page offset 90...
      Found 30 records on offset 90
    Fetching page offset 120...
      Found 30 records on offset 120
    Fetching page offset 150...
      Found 30 records on offset 150
    Fetching page offset 180...
      Found 30 records on offset 180
    Fetching page offset 210...
      Found 30 records on offset 210
    Fetching page offset 240...
      Found 30 records on offset 240
    Fetching page offset 270...
      Found 30 records on offset 270
    Fetching page offset 300...
      Found 30 records on offset 300
    Fetching page offset 330...
      Found 30 records on offset 330

In [ ]:
import json

file_path = "/content/New York City_Attraction.json"

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

if data:
    print("First record from the file:")
    print(json.dumps(data[0], indent=2, ensure_ascii=False))
else:
    print("The file is empty or does not contain any records.")

In [ ]:
import random
import string
import json
import csv
import os
import math

page_size = 30

categories = {
    'Attraction': 'true',
    'Spas & Wellness': '40',
    'Museums': '49',
    'Nightlife': '20',
    'Sights & Landmarks': '47',
    'Fun & Games': '56',
    'Nature & Parks': '57',
    'Classes & Workshops': '41',
    'Boat Tours & Water Sports': '55',
    'Casinos & Gambling': '53',
    'Water & Amusement Parks': '52',
    'Zoos & Aquariums': '48'
}

# ── 1. Record extractor ──────────────────────────────────────────────────────

def extract_record(record, category_name):
    """Extract important fields from a WebPresentation_SingleFlexCardSection record."""
    card = record.get("singleFlexCardContent", {})

    # Basic info
    title       = (card.get("cardTitle") or {}).get("text")
    ordinal     = card.get("ordinalPrefix")
    primary_info = (card.get("primaryInfo") or {}).get("text")
    description = (card.get("descriptiveText") or {}).get("text", {})
    description = description.get("text") if isinstance(description, dict) else None

    # Rating
    bubble      = card.get("bubbleRating") or {}
    rating      = bubble.get("rating")
    review_count = bubble.get("reviewCount")

    # Badge
    badge       = card.get("badge") or {}
    badge_type  = badge.get("type")
    badge_year  = badge.get("year")

    # URL
    card_link   = (card.get("cardLink") or {}).get("webRoute") or {}
    url         = card_link.get("webLinkUrl")
    detail_id   = ((card.get("cardLink") or {}).get("webRoute") or {}) \
                    .get("typedParams", {}).get("detailId")

    # Geo from jsonLd
    latitude, longitude = None, None
    json_ld_raw = card.get("jsonLd")
    if json_ld_raw:
        try:
            json_ld = json.loads(json_ld_raw)
            geo = json_ld.get("geo", {})
            latitude  = geo.get("latitude")
            longitude = geo.get("longitude")
        except Exception:
            pass

    # Photo
    card_photo  = card.get("cardPhoto") or {}
    photo_url   = (card_photo.get("sizes") or {}).get("urlTemplate")

    # Tags
    tags = ", ".join(
        (t.get("text") or {}).get("text", "")
        for t in (card.get("tagIdInfo") or [])
    )

    # See more experiences link
    poi_product   = card.get("poiAttachedProduct") or {}
    see_more_link = (poi_product.get("seeMoreLink") or {})
    see_more_text = (see_more_link.get("text") or {}).get("text")
    see_more_url  = (see_more_link.get("webRoute") or {}).get("webLinkUrl")

    return {
        "category":        category_name,
        "detail_id":       detail_id,
        "title":           title,
        "ordinal":         ordinal,
        "primary_info":    primary_info,
        "tags":            tags,
        "rating":          rating,
        "review_count":    review_count,
        "badge_type":      badge_type,
        "badge_year":      badge_year,
        "description":     description,
        "latitude":        latitude,
        "longitude":       longitude,
        "url":             f"https://www.tripadvisor.com{url}" if url else None,
        "photo_url":       photo_url,
        "see_more_text":   see_more_text,
        "see_more_url":    f"https://www.tripadvisor.com{see_more_url}" if see_more_url else None,
    }

# ── 2. CSV writer ────────────────────────────────────────────────────────────

CSV_FIELDS = [
    "category", "detail_id", "title", "ordinal", "primary_info", "tags",
    "rating", "review_count", "badge_type", "badge_year", "description",
    "latitude", "longitude", "url", "photo_url", "see_more_text", "see_more_url"
]

def save_csv(rows, path):
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        writer.writeheader()
        writer.writerows(rows)

# ── 3. Load cities ───────────────────────────────────────────────────────────

with open("/content/150-cities-locationId.json", "r", encoding="utf-8") as f:
    cities_data = json.load(f)

# ── 4. Summary report ────────────────────────────────────────────────────────

summary = {}   # { localized_name: { category_name: count, ... } }

# ── 5. Main loop ─────────────────────────────────────────────────────────────

for city_name, city_info in cities_data.items():
    geo_id         = city_info["location_id"]
    localized_name = city_info.get("localizedName", city_name)

    # Safe folder name
    safe_city = localized_name.replace("/", "-")
    city_folder = os.path.join("/content", safe_city)
    os.makedirs(city_folder, exist_ok=True)

    summary[localized_name] = {}
    all_city_rows = []   # accumulates extracted rows across all categories for one CSV per city

    print(f"\n{'#'*60}")
    print(f"City: {localized_name}  (geo_id: {geo_id})")
    print(f"{'#'*60}")

    for category_name, category_value in categories.items():
        print(f"\n  {'='*50}")
        print(f"  Category: {category_name}")
        print(f"  {'='*50}")

        all_raw_records   = []   # raw JSON records for this category
        all_extracted_rows = []  # extracted dicts for CSV
        page = 0

        filters = [] if category_value == 'true' else [
            {"id": "category", "value": [category_value]}
        ]

        while True:
            print(f"    Fetching page offset {page}...")

            payload2 = {
                "variables": {
                    "request": {
                        "tracking": {
                            "screenName": "AttractionsFusion",
                            "pageviewUid": "333333ba-0c8e-4165-80f0-246118a6e5ef"
                        },
                        "routeParameters": {
                            "geoId": geo_id,
                            "filters": filters,
                            "contentType": "attraction",
                            "webVariant": "AttractionsFusion",
                            "pagee": str(page)
                        }
                    },
                    "sessionId": "239BCDD736C812FFBA1007D05B2D7008",
                    "unitLength": "MILES",
                    "currency": "USD",
                    "mapSurface": False,
                    "debug": False,
                    "polling": False
                },
                "extensions": {
                    "preRegisteredQueryId": "0a9ecd5c0d013d56"
                }
            }

            random_request_id = "".join(
                random.choice(string.ascii_lowercase + string.digits) for i in range(180)
            )
            headers = {
                "X-Requested-By": random_request_id,
                "Referer": "https://www.tripadvisor.com/",
                "Origin": "https://www.tripadvisor.com",
            }

            try:
                result = await client.post(
                    url="https://www.tripadvisor.com/data/graphql/ids",
                    json=payload2,
                    headers=headers,
                )
                data = json.loads(result.content)
                sections = data['data']['Result'][0]['sections']
                filtered_records = [
                    r for r in sections
                    if r.get('__typename') == "WebPresentation_SingleFlexCardSection"
                ]

                print(f"      Found {len(filtered_records)} records on offset {page}")

                all_raw_records.extend(filtered_records)
                for rec in filtered_records:
                    all_extracted_rows.append(extract_record(rec, category_name))

                if len(filtered_records) < page_size:
                    print(f"      Last page ({len(filtered_records)} < {page_size}). Stopping.")
                    break

                page += page_size

            except Exception as e:
                print(f"      ERROR on offset {page}: {e}")
                break

        # ── Save per-category JSON inside city folder ──────────────────────
        safe_cat = category_name.replace(' ', '_').replace('&', 'and').replace('/', '-')
        json_path = os.path.join(city_folder, f"{localized_name}_{safe_cat}.json")
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(all_raw_records, f, indent=2, ensure_ascii=False)

        # ── Save per-category CSV inside city folder ───────────────────────
        csv_path = os.path.join(city_folder, f"{localized_name}_{safe_cat}.csv")
        save_csv(all_extracted_rows, csv_path)

        # ── Accumulate for city-wide CSV ───────────────────────────────────
        all_city_rows.extend(all_extracted_rows)

        # ── Update summary ─────────────────────────────────────────────────
        summary[localized_name][category_name] = len(all_raw_records)
        print(f"  Saved {len(all_raw_records)} records → {json_path}")

    # ── Save combined CSV for the whole city ──────────────────────────────
    city_csv_path = os.path.join(city_folder, f"{localized_name}_all_categories.csv")
    save_csv(all_city_rows, city_csv_path)
    print(f"\n  Combined CSV saved → {city_csv_path}")

# ── 6. Save summary report ───────────────────────────────────────────────────

summary_path = "/content/summary_report.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"\n✅ All done! Summary saved → {summary_path}")
print(json.dumps(summary, indent=2, ensure_ascii=False))


############################################################
City: New York City  (geo_id: 60763)
############################################################

  Category: Attraction
    Fetching page offset 0...
      Found 30 records on offset 0
    Fetching page offset 30...
      Found 30 records on offset 30
    Fetching page offset 60...
      Found 30 records on offset 60
    Fetching page offset 90...
      Found 30 records on offset 90
    Fetching page offset 120...
      Found 30 records on offset 120
    Fetching page offset 150...
      Found 30 records on offset 150
    Fetching page offset 180...
      Found 30 records on offset 180
    Fetching page offset 210...
      Found 30 records on offset 210
    Fetching page offset 240...
      Found 30 records on offset 240
    Fetching page offset 270...
      Found 30 records on offset 270
    Fetching page offset 300...
      Found 30 records on offset 300
    Fetching page offset 330...
      Found 30 records on offset 33

In [ ]:
df_attraction = pd.read_csv('/content/New York City/New York City_Attraction.csv')
display(df_attraction.head())

,category,detail_id,title,ordinal,primary_info,tags,rating,review_count,badge_type,badge_year,description,latitude,longitude,url,photo_url,see_more_text,see_more_url
0,Attraction,105127,Central Park,1.0,Points of Interest & Landmarks • Parks,"Points of Interest & Landmarks, Parks",4.7,134448.0,BEST_OF_BEST,2026.0,"For more than 150 years, visitors have flocked...",40.782750,-73.965600,https://www.tripadvisor.com/Attraction_Review-...,https://dynamic-media-cdn.tripadvisor.com/medi...,See ways to experience (131),https://www.tripadvisor.com/ClientLink?value=d...
1,Attraction,105125,The Metropolitan Museum of Art,2.0,Points of Interest & Landmarks • Art Museums,"Points of Interest & Landmarks, Art Museums",4.8,55533.0,BEST_OF_BEST,2026.0,At New York City's most visited museum and att...,40.779430,-73.963240,https://www.tripadvisor.com/Attraction_Review-...,https://dynamic-media-cdn.tripadvisor.com/medi...,See ways to experience (104),https://www.tripadvisor.com/ClientLink?value=b...
2,Attraction,1687489,The National 9/11 Memorial & Museum,3.0,Speciality Museums • Historic Sites,"Speciality Museums, Historic Sites",4.7,99119.0,BEST_OF_BEST,2026.0,"Through commemoration, exhibitions and educati...",40.711510,-74.013320,https://www.tripadvisor.com/Attraction_Review-...,https://dynamic-media-cdn.tripadvisor.com/medi...,See ways to experience (169),https://www.tripadvisor.com/ClientLink?value=d...
3,Attraction,104365,Empire State Building,4.0,Points of Interest & Landmarks • Architectural...,"Points of Interest & Landmarks, Architectural ...",4.5,97288.0,BEST_OF_BEST,2026.0,The Empire State Building is the World's Most ...,40.748400,-73.985700,https://www.tripadvisor.com/Attraction_Review-...,https://dynamic-media-cdn.tripadvisor.com/medi...,See ways to experience (61),https://www.tripadvisor.com/ClientLink?value=c...
4,Attraction,587661,Top of the Rock,5.0,Lookouts • Observation Decks & Towers,"Lookouts, Observation Decks & Towers",4.6,81386.0,NaN,NaN,"Top of the Rock Observation Deck, the newly op...",40.759003,-73.979324,https://www.tripadvisor.com/Attraction_Review-...,https://dynamic-media-cdn.tripadvisor.com/medi...,See ways to experience (29),https://www.tripadvisor.com/ClientLink?value=Q...
